In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [29]:
cell_repo = '/content/drive/MyDrive//Master Thesis/Code_trial_1/B-ALL_expression_matrix.csv'

# import data

with open(cell_repo, 'r') as f:
    lines = f.readlines()
# transform data to be in the right format
raw_df = pd.read_csv(cell_repo, sep = ';', decimal = ',')
df = raw_df.copy()


In [30]:
df = raw_df.drop(columns = ['Original_ID'])

# check number of healthy and blast cells
blast_cells = (df['IsBlast'] == 1).sum()
healthy_cells = (df['IsBlast'] == 0).sum()
blast_perc = blast_cells / len(df)

print(f'Number of Blast Cells: {blast_cells}')
print(f'Number of Healthy Cells: {healthy_cells}')
print(f'% of Blast Cells: {(blast_perc *100):.5f}%')





Number of Blast Cells: 2390
Number of Healthy Cells: 476610
% of Blast Cells: 0.49896%


In [31]:
import random
random.seed(1234)

def dataset_split(df, n_cells = 20000, blast_perc = 0.005, n_samples = 20):
    df = copy.deepcopy(df)

    blast_df = df[df['IsBlast']==1] # extract blast cells
    blast_df = blast_df.drop(columns = ['IsBlast'])

    healthy_df = df[df['IsBlast']==0] #extract healthy cells
    healthy_df = healthy_df.drop(columns = ['IsBlast'])

    #### healthy chunk division ####
    tot_healthy_cells = len(healthy_df)
    healthy_chunk_dim = int(tot_healthy_cells / n_samples) # define chunks dimension
    healthy_df_chunks = []
    counter = 0

    healthy_df = healthy_df.sample(frac=1, random_state=42)
    for i in range(n_samples): # divides healthy df in chunks (in this case 20 chunks since we are building 10 healthy and 10 blasted datasets)
        sample_i = healthy_df.iloc[counter: counter + healthy_chunk_dim]
        counter += healthy_chunk_dim
        healthy_df_chunks.append(sample_i)

    #### Blast chunk division ####
    tot_blast_cells = len(blast_df)
    blast_chunk_dim = int(tot_blast_cells / int(n_samples/2))
    blast_df_chunks = []
    counter = 0

    blast_df = blast_df.sample(frac=1, random_state=42)
    for i in range(int(n_samples/2)): # divides blast df in chunks (in tis case only 10 chunks)
        sample_i = blast_df.iloc[counter: counter + blast_chunk_dim]
        counter += blast_chunk_dim
        blast_df_chunks.append(sample_i)

    ### Healthy and Blast Samples ###
    all_samples = [] # stores all samples for citrus
    all_samples_labels = [] # stores all labels associated to samples

    healthy_dataset_1 = [] # stores healthy samples
    healthy_y_1 = []

    blast_dataset_1 = [] # stores samples with blast cells
    blast_y_1 = []

    blast_n = int(blast_perc*n_cells) # number of blast cells to add artificially
    healthy_n = n_cells - blast_n


    # create different healthy donor samples, artificially
    for i in range(int(n_samples/2)):
        healthy_chunk = healthy_df_chunks[i]
        healthy_sample = healthy_chunk.sample(n = n_cells, replace=False)
        healthy_dataset_1.append(healthy_sample)
        healthy_y_1.append(0)

    # create different non-healthy donor samples artificially
    for i in range(int(n_samples/2)):
        healthy_chunk = healthy_df_chunks[i+int(n_samples/2)]
        blast_chunk = blast_df_chunks[i]

        healthy_b_sample = healthy_chunk.sample(n = healthy_n, replace=False)
        blast_h_sample = blast_chunk.sample(n = blast_n, replace=False)

        blast_dataset_1.append(pd.concat([healthy_b_sample, blast_h_sample]))
        blast_y_1.append(1)


    ### Training, Validation and Test Splitting ###

    # sample indexes for train dataset
    dataset_idx = list(range(len(healthy_dataset_1)))
    train_perc = int(0.7*len(healthy_dataset_1))

    healthy_train_idx = random.sample(dataset_idx, train_perc)
    blast_train_idx = random.sample(dataset_idx, train_perc)

    # sample indexes for test dataset
    healthy_test_idx = list(set(dataset_idx) - set(healthy_train_idx))
    blast_test_idx = list(set(dataset_idx) - set(blast_train_idx))

    # sample indexes for val dataset
    val_perc = int(0.2*len(healthy_train_idx))

    healthy_val_idx = random.sample(healthy_train_idx, val_perc)
    blast_val_idx = random.sample(blast_train_idx, val_perc)

    healthy_train_idx = list(set(healthy_train_idx) - set(healthy_val_idx)) #subtract validation indexes from train indexes
    blast_train_idx = list(set(blast_train_idx) - set(blast_val_idx))

    # extract datasets and labels using indexes
    train_dataset_1 = []
    train_y_1 = []
    for i, j in zip(healthy_train_idx, blast_train_idx):
        train_dataset_1.append(healthy_dataset_1[i])
        train_dataset_1.append(blast_dataset_1[j])
        train_y_1.append(healthy_y_1[i])
        train_y_1.append(blast_y_1[j])

        all_samples.append(healthy_dataset_1[i])
        all_samples.append(blast_dataset_1[j])
        all_samples_labels.append(healthy_y_1[i])
        all_samples_labels.append(blast_y_1[j])

    val_dataset_1 = []
    val_y_1 = []
    for i, j in zip(healthy_val_idx, blast_val_idx):
        val_dataset_1.append(healthy_dataset_1[i])
        val_dataset_1.append(blast_dataset_1[j])
        val_y_1.append(healthy_y_1[i])
        val_y_1.append(blast_y_1[j])

        all_samples.append(healthy_dataset_1[i])
        all_samples.append(blast_dataset_1[j])
        all_samples_labels.append(healthy_y_1[i])
        all_samples_labels.append(blast_y_1[j])

    test_dataset_1 = []
    test_y_1 = []
    for i,j in zip(healthy_test_idx, blast_test_idx):
        test_dataset_1.append(healthy_dataset_1[i])
        test_dataset_1.append(blast_dataset_1[j])
        test_y_1.append(healthy_y_1[i])
        test_y_1.append(blast_y_1[j])

        all_samples.append(healthy_dataset_1[i])
        all_samples.append(blast_dataset_1[j])
        all_samples_labels.append(healthy_y_1[i])
        all_samples_labels.append(blast_y_1[j])

    return train_dataset_1, train_y_1, val_dataset_1, val_y_1, test_dataset_1, test_y_1, all_samples, all_samples_labels


In [32]:
# Let us convert the list of pandas dataframs in an array
import copy
import numpy as np
def df_to_array(df):
    array = []
    for sample in df:
      array.append(sample.values)
    return array


train_dataset, train_y, val_dataset, val_y, test_dataset, test_y, all_samples, all_samples_labels = dataset_split(df,20000, 0.005, 20) # build train, test and val datasets

# Train
train_samples_array = df_to_array(train_dataset)
train_y_array = np.array(train_y)

# Validation
val_samples_array = df_to_array(val_dataset)
val_y_array = np.array(val_y)

#Test
test_samples_array = df_to_array(test_dataset)
test_y_array = np.array(test_y)

print(len(train_samples_array[0]))

daa = train_samples_array[0]
print(daa)
'''
no_head_train = []
for dataset in train_dataset:
    #dataset.iloc[1:]
    no_head_train.append(dataset.iloc[1:])

print(no_head_train[0])
'''
#print(isinstance(train_dataset, list))

20000
[[0.84600794 0.9671433  0.49374247 ... 1.24065685 3.12258649 2.71802425]
 [0.93700707 1.10336697 0.52464932 ... 1.80334294 0.30346817 2.60919714]
 [0.75219363 0.87881565 0.45368806 ... 0.56326699 0.3755883  2.62825036]
 ...
 [0.87471384 1.10468876 0.4917469  ... 1.3451345  0.0648419  2.72939682]
 [0.69027483 0.80681616 0.45512483 ... 1.54818952 2.99784827 2.79294109]
 [0.76681006 0.92005074 0.48400423 ... 1.68790829 0.18157864 2.56903052]]


'\nno_head_train = []\nfor dataset in train_dataset:\n    #dataset.iloc[1:]\n    no_head_train.append(dataset.iloc[1:])\n\nprint(no_head_train[0])\n'

In [33]:
#print(train_dataset[0])

xt = train_dataset[0]
xt = np.array(xt)
print(xt)

#for x in train_y:
#  print(isinstance(x, int))

[[0.84600794 0.9671433  0.49374247 ... 1.24065685 3.12258649 2.71802425]
 [0.93700707 1.10336697 0.52464932 ... 1.80334294 0.30346817 2.60919714]
 [0.75219363 0.87881565 0.45368806 ... 0.56326699 0.3755883  2.62825036]
 ...
 [0.87471384 1.10468876 0.4917469  ... 1.3451345  0.0648419  2.72939682]
 [0.69027483 0.80681616 0.45512483 ... 1.54818952 2.99784827 2.79294109]
 [0.76681006 0.92005074 0.48400423 ... 1.68790829 0.18157864 2.56903052]]


In [34]:

import sys
import os
import importlib
# Aggiungi la directory principale del progetto al percorso di sistema
path = '/content/drive/MyDrive/Master Thesis/CSNN/csnn'
if path not in sys.path:
    sys.path.append(path)
# Ricaricali per forzare la lettura aggiornata del file


# Importa tutte le funzioni e gli oggetti dal modulo trainer
# Attenzione: l'uso di '*' non è sempre raccomandato in codice di produzione
# perché rende meno chiaro da dove provengano le funzioni.
from utils import trainer, DensityDifference

importlib.reload(trainer)
importlib.reload(DensityDifference)

from utils import trainer, DensityDifference


In [10]:
from tqdm import tqdm
import numpy as np

x_train = np.arange(10)        # finto dataset
y_train = np.array([0,1,0,1,0,1,0,0,1,0])

for x in tqdm(x_train[y_train == 1], disable=False):
    print(f'\n')
    print(x)

100%|██████████| 4/4 [00:00<00:00, 22764.20it/s]



1


3


5


8


In [36]:
alpha = 1
dd_sample_size = 10
# sets parameter for KDE class
dd = DensityDifference.DensityDifference(alpha, sample_size=dd_sample_size)

# Train the KDE
allpos_sample, allneg_sample = dd.fit(train_samples_array, train_y)

1
2
3
4
1
2
3
4
1
2
3
4
1
2
3
4
1
2
3
4
1
2
3
4
ok

ok


  0%|          | 0/20000 [00:00<?, ?it/s]

1
Iteration 0: [0.84600794 0.9671433  0.49374247 2.10809278 0.64026421 1.45579839
 2.82005382 0.69655949 1.24065685 3.12258649 2.71802425]

ok


IndexError: too many indices for array: array is 1-dimensional, but 2 were indexed

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
'''
import shutil
import os

def copy_directory_to_drive():
    source_dir = "/content/csnn"
    dest_dir = "/content/drive/MyDrive/Master Thesis/CSNN"

    print(f"Copia di {source_dir} in {dest_dir}...")

    try:
        # Crea la directory di destinazione se non esiste
        os.makedirs(dest_dir, exist_ok=True)

        # Copia i contenuti (se la cartella di destinazione esiste già, shutil.copytree fallisce)
        if os.path.exists(os.path.join(dest_dir, "csnn")):
            print("La cartella csnn esiste già nella destinazione. Eliminando la versione precedente...")
            shutil.rmtree(os.path.join(dest_dir, "csnn"))

        shutil.copytree(source_dir, os.path.join(dest_dir, "csnn"))
        print("Copia completata.")

    except Exception as e:
        print(f"Errore durante la copia: {e}")

# Esegui la funzione
if __name__ == "__main__":
    copy_directory_to_drive()
'''

In [ ]:
# ========================================
# IMPORT CSNN ORIGINALE SU GOOGLE COLAB
# ========================================

# 1. SETUP INIZIALE
print("=== SETUP CSNN SU GOOGLE COLAB ===")

# Clona la repository
!git clone https://github.com/erobl/csnn.git
%cd csnn

# Mostra struttura del progetto
print("\n📁 STRUTTURA PROGETTO:")
!tree -L 2

# 2. INSTALLAZIONE DIPENDENZE
print("\n=== INSTALLAZIONE DIPENDENZE ===")

# Mostra i requirements
print("📋 Requirements originali:")
!cat requirements.txt

# Installa i requirements
!pip install -r requirements.txt

# Verifica versioni installate
print("\n✅ VERIFICA INSTALLAZIONE:")
import torch
import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
import sklearn
from tqdm import tqdm

print(f"✓ PyTorch: {torch.__version__}")
print(f"✓ NumPy: {np.__version__}")
print(f"✓ Pandas: {pd.__version__}")
print(f"✓ Scikit-learn: {sklearn.__version__}")
print(f"✓ CUDA disponibile: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✓ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# 3. ESPLORAZIONE CODICE
print("\n=== ESPLORAZIONE CODICE ===")

# Mostra file Python principali
print("📝 FILE PYTHON PRINCIPALI:")
!ls -la *.py

# Mostra configurazioni
print("\n⚙️ FILE CONFIGURAZIONE:")
!ls -la config/

# Mostra alcune configurazioni di esempio
print("\n📄 Esempio configurazione B-ALL:")
!head -20 config/best_ball_logistic.yaml

# 4. ANALISI STRUTTURA MODULI
print("\n=== ANALISI MODULI ===")

# Controlla se ci sono moduli custom
import os
import importlib.util

def analyze_python_files():
    python_files = [f for f in os.listdir('.') if f.endswith('.py')]

    for file in python_files:
        print(f"\n📄 {file}:")
        # Mostra prime righe per capire cosa fa
        with open(file, 'r') as f:
            lines = f.readlines()[:10]
            for i, line in enumerate(lines, 1):
                if line.strip().startswith('#') or line.strip().startswith('"""'):
                    print(f"  {i}: {line.rstrip()}")
                elif 'import' in line or 'def ' in line or 'class ' in line:
                    print(f"  {i}: {line.rstrip()}")

analyze_python_files()

# 5. CONTROLLO DIPENDENZE DEL MODELLO
print("\n=== CONTROLLO MODELLO ===")

# Cerca i file che definiscono il modello CSNN
!grep -l "class.*CSNN\|def.*csnn" *.py

# Cerca le funzioni di training
!grep -l "train.*" *.py | head -5

# 6. VERIFICA REQUISITI MINIMI
print("\n=== VERIFICA SISTEMA ===")

import psutil
import sys

# RAM
ram = psutil.virtual_memory()
print(f"💾 RAM: {ram.available/1e9:.1f}/{ram.total/1e9:.1f} GB disponibili")

# CPU
print(f"🖥️ CPU cores: {psutil.cpu_count()}")

# Python
print(f"🐍 Python: {sys.version}")

# Spazio disco
disk = psutil.disk_usage('.')
print(f"💿 Spazio disco: {disk.free/1e9:.1f} GB liberi")

# 7. TEST IMPORT MODULI
print("\n=== TEST IMPORT MODULI ===")

try:
    # Prova ad importare moduli del progetto se esistono
    sys.path.append('.')

    # Lista dei possibili moduli
    possible_modules = ['model', 'utils', 'data_loader', 'train', 'csnn']

    for module_name in possible_modules:
        if os.path.exists(f"{module_name}.py"):
            try:
                module = importlib.import_module(module_name)
                print(f"✓ {module_name}.py importato con successo")

                # Mostra le funzioni/classi principali
                attrs = [attr for attr in dir(module) if not attr.startswith('_')]
                if attrs:
                    print(f"  Contiene: {', '.join(attrs[:5])}")

            except Exception as e:
                print(f"❌ Errore import {module_name}: {str(e)[:50]}...")

except Exception as e:
    print(f"Errore generale: {e}")

# 8. PREPARAZIONE PER STUDIO
print("\n=== PREPARAZIONE STUDIO CODICE ===")

# Crea notebook helper functions
def show_file_content(filename, lines=None):
    """Mostra contenuto di un file"""
    if os.path.exists(filename):
        with open(filename, 'r') as f:
            content = f.readlines()
            if lines:
                content = content[:lines]
            for i, line in enumerate(content, 1):
                print(f"{i:3}: {line.rstrip()}")
    else:
        print(f"File {filename} non trovato")

def find_function(func_name):
    """Trova una funzione nei file Python"""
    python_files = [f for f in os.listdir('.') if f.endswith('.py')]

    for file in python_files:
        with open(file, 'r') as f:
            lines = f.readlines()
            for i, line in enumerate(lines):
                if f"def {func_name}" in line or f"class {func_name}" in line:
                    print(f"📍 Trovato in {file}:{i+1}")
                    print(f"   {line.strip()}")

def list_configs():
    """Lista tutte le configurazioni disponibili"""
    if os.path.exists('config'):
        configs = [f for f in os.listdir('config') if f.endswith('.yaml')]
        for config in configs:
            print(f"⚙️ {config}")

print("🔧 FUNZIONI HELPER DISPONIBILI:")
print("- show_file_content('filename.py', lines=20)")
print("- find_function('function_name')")
print("- list_configs()")

list_configs()

# 9. PRIMO TEST DI FUNZIONAMENTO
print("\n=== PRIMO TEST ===")

# Prova a vedere se riesce a caricare una configurazione
try:
    config_file = 'config/best_ball_logistic.yaml'
    if os.path.exists(config_file):
        with open(config_file, 'r') as f:
            config = yaml.safe_load(f)
        print(f"✓ Configurazione {config_file} caricata:")
        print(f"  Chiavi principali: {list(config.keys())}")
    else:
        print("❌ File configurazione non trovato")

except Exception as e:
    print(f"❌ Errore caricamento config: {e}")

print("\n🎉 SETUP COMPLETATO!")
print("Ora puoi studiare il codice usando:")
print("- show_file_content('train_logistic.py') per vedere il training")
print("- find_function('CSNN') per trovare la definizione del modello")
print("- !python train_logistic.py --help per vedere le opzioni")

In [ ]:
# Vedi tutti i file Python
!ls -la *.py

# Vedi la struttura generale
!head -10 *.py

In [ ]:
# Cerca KDE (Kernel Density Estimation)
!grep -r -A 5 -B 5 "kde\|kernel\|density" *.py

print('ok')
# Cerca il calcolo di Bayes
!grep -r -A 10 "bayes\|probability\|P(" *.py